# 02 - Analyse des KPI industriels

Objectif : analyser les indicateurs de performance, comparer les lignes de production et formuler des recommandations métier.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT_DIR / 'data' / 'processed' / 'donnees_production_nettoyees.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['date_production', 'annee_mois'], encoding='utf-8')
df.head()

In [ ]:
kpi_globaux = {
    'taux_conformite': df['quantite_conforme'].sum() / df['quantite_produite'].sum(),
    'taux_non_conformite': df['quantite_non_conforme'].sum() / df['quantite_produite'].sum(),
    'taux_retard_livraison': df['retard_livraison'].mean(),
    'cout_non_qualite': df['cout_non_qualite'].sum(),
    'temps_cycle_moyen_min': df['temps_cycle_min'].mean(),
    'disponibilite_machine': df['taux_disponibilite_machine'].mean(),
    'anomalies_critiques': int(df['anomalie_critique'].sum()),
}
pd.Series(kpi_globaux)

In [ ]:
mensuel = df.groupby('annee_mois').agg(
    quantite_produite=('quantite_produite', 'sum'),
    quantite_conforme=('quantite_conforme', 'sum'),
    cout_non_qualite=('cout_non_qualite', 'sum'),
    taux_retard=('retard_livraison', 'mean')
).reset_index()
mensuel['taux_conformite'] = mensuel['quantite_conforme'] / mensuel['quantite_produite']

px.line(mensuel, x='annee_mois', y='taux_conformite', markers=True, title='Évolution mensuelle du taux de conformité')

In [ ]:
par_ligne = df.groupby('ligne_production').agg(
    quantite_produite=('quantite_produite', 'sum'),
    taux_retard=('retard_livraison', 'mean'),
    cout_non_qualite=('cout_non_qualite', 'sum'),
    productivite=('productivite_par_heure', 'mean'),
    disponibilite=('taux_disponibilite_machine', 'mean')
).reset_index()
par_ligne.sort_values('cout_non_qualite', ascending=False)

In [ ]:
anomalies = df[df['type_anomalie'] != 'Aucune'].groupby('type_anomalie').agg(
    nombre=('type_anomalie', 'size'),
    cout_non_qualite=('cout_non_qualite', 'sum'),
    anomalies_critiques=('anomalie_critique', 'sum')
).reset_index().sort_values('cout_non_qualite', ascending=False)
anomalies

## Recommandations

- Prioriser les lignes qui cumulent retard, coût de non-qualité et disponibilité machine dégradée.
- Suivre les anomalies critiques dans un tableau opérationnel quotidien.
- Définir des seuils d'alerte sur le taux de conformité et le taux de retard.
- Compléter l'analyse par une revue terrain avec les équipes qualité et production.